# 03 - Model vs NWS flood warnings (K-fold CV)

Benchmarks the trained GOES/GLM model against **NWS Flash-Flood + Areal-Flood WARNINGS** -
the operational forecaster baseline. Both are rasterized to the 50 km grid on the **fixed
test set** (the held-out CV test months, spanning all of 2019-2025) and scored against the
**observed-flood labels** (Groundsource union NCEI storm events - the same `y` the model was
trained on).

The model prediction is the **K-fold ensemble**: each model is trained once per CV fold
(`foldsplit.py`), and its test prediction is the mean of the fold-models' probabilities on
the fixed test set. The operating threshold is the best-F1 point on the **pooled CV
validation** (each fold's own val, which together partition the non-test pool).

Note on fairness: NWS warnings are same-day, short-lead nowcasts, while our model predicts
day D from **D-1** GOES (a ~1-day lead). So this is "who better flags the cells that flood",
not a like-for-like lead comparison - the model is doing a strictly harder (longer-lead) task.

Sections: (1) model predictions, (2) warnings -> 50 km grid, (3) metrics table
(model & warnings vs observed), (4) who catches the observed
floods, (5) per-day maps, (6) aggregate spatial view, (7) metric bar chart.


## 0. Setup

In [ ]:
import os
import pickle
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import average_precision_score, precision_recall_curve
from torch.utils.data import DataLoader

from floodlens import foldsplit
from floodlens.config import STATES_GEOJSON, UNIFIED_PARQUET, build_grid_cells, model_artifact
from floodlens.gridindex import build_pix2cell
import floodlens.trainers.convgru_attn as convgru_attn  # noqa: E402

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
DEEP = {"convgru_attn": convgru_attn}          # committed model (nb03 is ConvGRU+attn only)
N_FOLDS = foldsplit.N_FOLDS

_, GRID_R, GRID_C, land = build_pix2cell()
CACHE_DIR = convgru_attn.CACHE_DIR


def fold_days(k):
    """Set the fold and return (train, val, test); test is fixed across folds."""
    os.environ["FOLD"] = str(k)
    return foldsplit.fold_splits(CACHE_DIR)


_, _, te_days = fold_days(0)                    # fixed held-out test set


def base_rate(days):
    y = np.stack([np.load(CACHE_DIR / f"{d}_y.npy") for d in days])
    return float(y[:, land].mean())


print(f"device {DEVICE} | grid {GRID_R}x{GRID_C} | land {int(land.sum())} cells | "
      f"{N_FOLDS}-fold CV")
print(f"fixed test: {len(te_days)} days  base rate {base_rate(te_days):.4f}")

# grid polygons + state outlines for maps + rasterizing warnings
cells_gdf, _, _, _ = build_grid_cells()
cells_albers = cells_gdf.to_crs(5070)
RR, CC = cells_albers["R"].to_numpy(), cells_albers["C"].to_numpy()
states = gpd.read_file(STATES_GEOJSON)
states = states[~states["name"].isin(["Alaska", "Hawaii", "Puerto Rico"])].to_crs(5070)


## 1. Model predictions (K-fold ensemble)

The model's test prediction is the **mean over CV folds** of each fold-model's probability on
the fixed test set. The operating threshold is the best-F1 point on the **pooled CV
validation** (every fold scored on its own val, concatenated). This is the committed
deliverable — the **6-fold ConvGRU+attn** ensemble.


In [ ]:
# The committed deliverable: the 6-fold ConvGRU+attn ensemble (mean of the per-fold
# checkpoints' probabilities on the fixed test set), thresholded on the pooled CV val.
MODEL = "convgru_attn"


def _fold_ckpt(name, k):
    ext = "pt" if name in DEEP else "pkl"
    return model_artifact(name, f"_f{k}", ext, make=False)


def _folds_avail(name):
    return [k for k in range(N_FOLDS) if _fold_ckpt(name, k).exists()]


@torch.no_grad()
def _infer_ckpt(name, ckpt, days):
    """(probs, trues) for one fold's checkpoint of one model."""
    if name in DEEP:
        mod = DEEP[name]
        net = mod.FloodNet().to(DEVICE)                   # stats are saved buffers
        net.load_state_dict(torch.load(ckpt, map_location=DEVICE))
        net.eval()
        probs, trues = [], []
        loader = DataLoader(mod.FeatureCache(days), batch_size=16, num_workers=8)
        for seq, summ, t, y in loader:
            with torch.autocast("cuda", dtype=torch.bfloat16):
                pr = torch.sigmoid(net(seq.to(DEVICE).float(), summ.to(DEVICE).float(),
                                       t.to(DEVICE).float())).squeeze(1)
            probs.append(pr.float().cpu().numpy())
            trues.append(y.numpy())
        del net
        torch.cuda.empty_cache()
        return np.concatenate(probs), np.concatenate(trues)
    raise ValueError(f"{name}: nb03 evaluates the ConvGRU+attn deep model only")


_CVCACHE = {}


def cv_model(name):
    """Per-model CV: fixed-test ensemble probs + pooled-val probs/trues (folds concat)."""
    if name not in _CVCACHE:
        tprobs, vpr, vtr = [], [], []
        for k in _folds_avail(name):
            _, va_k, _ = fold_days(k)
            vp, vy = _infer_ckpt(name, _fold_ckpt(name, k), va_k)   # fold-k own val
            tp, _ = _infer_ckpt(name, _fold_ckpt(name, k), te_days)  # fixed test
            vpr.append(vp); vtr.append(vy); tprobs.append(tp)
        _CVCACHE[name] = dict(test=np.mean(tprobs, axis=0),
                              vprobs=np.concatenate(vpr),
                              vtrues=np.concatenate(vtr))
    return _CVCACHE[name]


def spec_name(spec):
    if isinstance(spec, str):
        return spec
    return " + ".join(f"{w:g}*{n}" for n, w in spec.items())


def _exists(spec):
    names = [spec] if isinstance(spec, str) else list(spec)
    return all(_folds_avail(n) for n in names)


def infer_cv(spec):
    """spec = name or {name: weight}; -> (test_probs, val_probs, val_trues), CV-ensembled.

    Blends model fold-ensembles by weight on both the fixed test and the pooled CV val
    (fold order is identical across models, so the pooled-val arrays align)."""
    items = {spec: 1.0} if isinstance(spec, str) else spec
    wsum = sum(items.values())
    tp = vp = vy = None
    for name, w in items.items():
        cm = cv_model(name); f = w / wsum
        tp = f * cm["test"] if tp is None else tp + f * cm["test"]
        vp = f * cm["vprobs"] if vp is None else vp + f * cm["vprobs"]
        vy = cm["vtrues"]
    return tp, vp, vy


assert _exists(MODEL), f"{MODEL} not fully trained yet"
MODEL_NAME = spec_name(MODEL)

Y = np.stack([np.load(CACHE_DIR / f"{d}_y.npy") for d in te_days])   # observed test labels
probs, vp, vy = infer_cv(MODEL)
pr, rc, thr = precision_recall_curve(vy[:, land].ravel(), vp[:, land].ravel())
f1c = 2 * pr * rc / (pr + rc + 1e-9)
THR = float(thr[np.argmax(f1c[:-1])])
M = (probs >= THR).astype(np.float32)                   # binarized prediction (test)
ap = average_precision_score(Y[:, land].ravel(), probs[:, land].ravel())
print(f"model: {MODEL_NAME}  ({N_FOLDS}-fold ensemble)")
print(f"test AUPRC {ap:.4f}  |  pooled-val-F1 threshold {THR:.3f}  |  "
      f"flagged cells/day {M[:, land].sum() / len(te_days):.1f}")


## 2. NWS warnings -> 50 km grid

A cell is **warned** on CDT day D if an NWS Flash-Flood (FF) or Areal-Flood (FA) *warning*
polygon is active during that CDT day and intersects the cell. Warning issue/expire are UTC;
we shift by -5 h to CDT (matching the label construction) and rasterize onto the same grid,
for the test days only.

In [ ]:
CDT = pd.Timedelta(hours=5)
te_ts = pd.to_datetime(te_days, format="%Y%m%d")
te_set = set(te_ts)

u = gpd.read_parquet(UNIFIED_PARQUET)
w = u[u["source"].isin(["ff_warning", "fa_warning"])].copy()
w["issue_cdt"] = w["issue_date"] - CDT
w["expire_cdt"] = w["expire_date"] - CDT
# keep warnings that could touch a test CDT day (cheap pre-filter)
w = w[(w["expire_cdt"] >= te_ts.min()) & (w["issue_cdt"] <= te_ts.max() + pd.Timedelta(days=1))]


def cdt_days(issue, expire):
    d = pd.date_range(issue.normalize(), expire.normalize(), freq="D")
    return d if len(d) <= 5 else d[:1]


w["day"] = [cdt_days(i, e) for i, e in zip(w["issue_cdt"], w["expire_cdt"])]
ev = w.explode("day", ignore_index=True)
ev = ev[ev["day"].isin(te_set)]

j = gpd.sjoin(cells_gdf, ev[["day", "geometry"]], predicate="intersects")
warn_map = {}
for day, g in j.groupby("day"):
    a = np.zeros((GRID_R, GRID_C), np.float32)
    a[g["R"], g["C"]] = 1.0
    warn_map[day.strftime("%Y%m%d")] = a

W = np.stack([warn_map.get(d, np.zeros((GRID_R, GRID_C), np.float32)) for d in te_days])
print(f"warnings rasterized: {len(ev):,} warning-days -> "
      f"{sum(v.sum() > 0 for v in warn_map.values())}/{len(te_days)} test days have >=1 warning")
print(f"warned land cells/day: {W[:, land].sum() / len(te_days):.1f}  "
      f"(observed floods/day: {Y[:, land].sum() / len(te_days):.1f})")

## 3. Metrics - model & warnings vs observed floods

Both predictions scored against the observed labels on land cells: precision, recall, F1, CSI
(exact and 1-grid-tolerant). The model additionally has AUPRC (probabilistic); warnings are
binary. `xbase` = AUPRC / base rate.

In [ ]:
def _dilate(m):
    o = m.copy()
    o[:-1, :] |= m[1:, :]; o[1:, :] |= m[:-1, :]
    o[:, :-1] |= m[:, 1:]; o[:, 1:] |= m[:, :-1]
    o[:-1, :-1] |= m[1:, 1:]; o[1:, 1:] |= m[:-1, :-1]
    o[:-1, 1:] |= m[1:, :-1]; o[1:, :-1] |= m[:-1, 1:]
    return o


def binary_metrics(pred, y):
    """P/R/F1/CSI exact + 1-grid for a binary prediction (N,R,C) vs labels."""
    pb = pred[:, land].ravel().astype(int)
    t = y[:, land].ravel().astype(int)
    tp = int((pb * t).sum()); fp = int((pb * (1 - t)).sum()); fn = int(((1 - pb) * t).sum())
    prec = tp / (tp + fp + 1e-9); rec = tp / (tp + fn + 1e-9)
    f1 = 2 * prec * rec / (prec + rec + 1e-9); csi = tp / (tp + fn + fp + 1e-9)
    h = m1 = fa = 0
    for i in range(len(pred)):
        yt = (y[i] > 0.5) & land
        yp = (pred[i] > 0.5) & land
        h += int((yt & (_dilate(yp) & land)).sum())
        m1 += int((yt & ~(_dilate(yp) & land)).sum())
        fa += int((yp & ~(_dilate(yt) & land)).sum())
    p1 = h / (h + fa + 1e-9); r1 = h / (h + m1 + 1e-9)
    return dict(P=prec, R=rec, POD=rec, F1=f1, CSI=csi,
                F1_1=2 * p1 * r1 / (p1 + r1 + 1e-9), CSI1=h / (h + m1 + fa + 1e-9))


base = base_rate(te_days)
rows = []
mm = binary_metrics(M, Y)
mm["AUPRC"] = average_precision_score(Y[:, land].ravel(), probs[:, land].ravel())
mm["xbase"] = mm["AUPRC"] / base
rows.append(dict(predictor=f"{MODEL_NAME} (D-1 GOES)", **mm))
wm = binary_metrics(W, Y)
wm["AUPRC"] = float("nan"); wm["xbase"] = float("nan")
rows.append(dict(predictor="NWS warnings", **wm))

tbl = pd.DataFrame(rows).set_index("predictor")[
    ["AUPRC", "xbase", "P", "POD", "F1", "CSI", "F1_1", "CSI1"]]
print(f"test base rate {base:.4f}  ({len(te_days)} days)\n")
print(tbl.round(3).to_string())

## 4. Who catches the observed floods?

Of every observed flood cell-day on the test set, how many are flagged by the **model only**,
**NWS only**, **both**, or **neither** - and the same for false alarms (flagged but no observed
flood). This shows whether the model adds coverage beyond the operational warnings.

In [ ]:
# Coverage of observed floods + detection metrics for model, NWS, their union & intersection
import matplotlib.patches as mpatches
ml = M[:, land].astype(bool).ravel()
wl = W[:, land].astype(bool).ravel()
yl = Y[:, land].astype(bool).ravel()
ul = ml | wl                                                 # union: caught by model OR NWS
pos = int(yl.sum())
both = int((ml & wl & yl).sum()); mo = int((ml & ~wl & yl).sum())
wo = int((~ml & wl & yl).sum()); miss = int((~ml & ~wl & yl).sum())

MC, NC, UC, IC = "#3a86ff", "#56cfe1", "#9d4edd", "#03045e"  # model, NWS, union, intersection (colorblind-safe cool)
disp = "ConvGRU+Attn"


def _bm(pred):
    tp = int((pred & yl).sum()); fp = int((pred & ~yl).sum()); fn = int((~pred & yl).sum())
    P = tp / (tp + fp + 1e-9); R = tp / (tp + fn + 1e-9)
    return R, P, 2 * P * R / (P + R + 1e-9), tp / (tp + fp + fn + 1e-9)   # recall, prec, F1, CSI


mvals, wvals, uvals = _bm(ml), _bm(wl), _bm(ul)
# order: ConvGRU+Attn, NWS warnings, Model or NWS, Model and NWS
cov_labels = [disp, "NWS warnings", "Model or NWS", "Model and NWS"]
cov_num = [both + mo, both + wo, both + mo + wo, both]      # flooded cells detected
cov_vals = [n / pos * 100 for n in cov_num]
cov_cols = [MC, NC, UC, IC]
print(f"observed flood cell-days on test: {pos:,} ({len(te_days)} days)")
for lab, v in zip(cov_labels, cov_vals):
    print(f"  {lab.replace(chr(10), ' '):<28}: {v:5.1f}%")

fig, (axL, axR) = plt.subplots(1, 2, figsize=(16, 6.5), dpi=120,        # 1920 x 780 px (poster)
                               constrained_layout=True)
fig.patch.set_alpha(0)
axL.set_facecolor("none"); axR.set_facecolor("none")

# --- left: % of observed flood cell-days covered (vertical bars) ---
xs = np.arange(len(cov_labels))
bars = axL.bar(xs, cov_vals, color=cov_cols, edgecolor="white", width=0.72)
for b, v, n in zip(bars, cov_vals, cov_num):
    axL.text(b.get_x() + b.get_width() / 2, v + 1, f"{v:.1f}%",
             ha="center", va="bottom", fontsize=15, fontweight="bold")
    axL.text(b.get_x() + b.get_width() / 2, v / 2,
             f"$\\mathbf{{\\dfrac{{{n}}}{{{pos}}}}}$",
             ha="center", va="center", fontsize=17, fontweight="bold", color="white")
axL.set_xticks(xs); axL.set_xticklabels(cov_labels, fontsize=12.5)
axL.set_ylabel("% of observed flood cell-days covered", fontsize=14)
axL.set_ylim(0, max(cov_vals) * 1.15)
axL.set_title("Coverage of observed floods", fontsize=16, fontweight="bold")
axL.tick_params(labelsize=11); axL.spines[["top", "right"]].set_visible(False)

# --- right: detection metrics (recall / precision / F1 / CSI) ---
keys = ["Recall", "Precision", "F1", "CSI"]
x = np.arange(len(keys)); bw = 0.26
# order: ConvGRU+Attn, NWS warnings, Model or NWS (no "Model and NWS" here)
for off, v3, lab, col in [(-bw, mvals, f"{disp} (D-1)", MC),
                          (0.0, wvals, "NWS warnings", NC),
                          (bw, uvals, "Model or NWS", UC)]:
    axR.bar(x + off, v3, bw, label=lab, color=col)
    for xi, vi in zip(x, v3):
        axR.text(xi + off, vi + 0.008, f"{vi:.2f}", ha="center", fontsize=10.5,
                 fontweight="bold")
axR.set_xticks(x); axR.set_xticklabels(keys, fontsize=13)
axR.set_ylabel("score vs observed floods", fontsize=14)
axR.set_ylim(0, max(max(mvals), max(wvals), max(uvals)) * 1.30)
axR.set_title("Detection metrics", fontsize=16, fontweight="bold")
axR.legend(handles=[mpatches.Patch(color=MC, label=f"{disp} (D-1)"),
                    mpatches.Patch(color=NC, label="NWS warnings"),
                    mpatches.Patch(color=UC, label="Model or NWS"),
                    mpatches.Patch(color=IC, label="Model and NWS")],
           fontsize=15, frameon=False)
axR.spines[["top", "right"]].set_visible(False)

POSTER = Path("poster_figs"); POSTER.mkdir(exist_ok=True)
fig.savefig(POSTER / "4_coverage_and_metrics.png", dpi=120, transparent=True)  # 1920x780
plt.show()

## 5. Per-day maps

Picks two contrasting days from the **union of the test set and the 2026 held-out days** -
one where **our model beats NWS** and one where **NWS beats our model** - ranked by per-day
**CSI** advantage (CSI = hits / (hits + misses + false alarms)), among days above the **75th
percentile** of flooded-cell count. Each panel is a confusion map (hit / miss / false alarm)
for the model (top row) and NWS (bottom row); the day's source (test / 2026) is noted in its
title. The 2026 predictions are built here from the fold checkpoints, so this only needs
sections 1-2 to have run.

In [ ]:
# two contrasting days from the UNION of the test set + 2026 held-out days: the day the model
# beats NWS the most and the day NWS beats it the most, ranked by per-day CSI advantage.
# Layout: rows = the two winning days (each its own background tint), cols = the two forecasts.
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

LAND, MISS, HIT, FA = "#e9ecef", "#e63946", "#1b9e4b", "#f77f00"   # TN / FN(miss) / TP(hit) / FP(falsealarm)
ROW_BG = ["#e6eef7", "#f7efe6"]                                    # row 0 cool, row 1 warm
CMAP = ListedColormap([LAND, MISS, HIT, FA])                       # cat 0..3
LEG = [mpatches.Patch(fc=HIT, label="Hit (TP) - flood correctly flagged"),
       mpatches.Patch(fc=FA, label="False alarm (FP) - flagged, no flood"),
       mpatches.Patch(fc=MISS, label="Miss (FN) - flood not flagged"),
       mpatches.Patch(fc=LAND, ec="0.8", label="No flood & not flagged (TN)")]


def _day_csi(P, Yx):
    """Per-day CSI on land for a binary prediction (nan when nothing to score)."""
    out = np.full(len(Yx), np.nan)
    for i in range(len(Yx)):
        obs = (Yx[i] > 0) & land; prd = (P[i] > 0) & land
        h = int((obs & prd).sum()); m = int((obs & ~prd).sum()); f = int((prd & ~obs).sum())
        if h + m + f:
            out[i] = h / (h + m + f)
    return out


def _build_2026():
    """(days, Y, M, W) for the 2026 held-out set - standalone (reuses sec 1-2 helpers)."""
    man = pd.read_parquet(CACHE_DIR / "manifest.parquet")
    man["label_day"] = pd.to_datetime(man["label_day"])
    dd = [d.strftime("%Y%m%d") for d in man.loc[man.label_day.dt.year == 2026, "label_day"]]
    dd = [d for d in dd if (CACHE_DIR / f"{d}_sum.npy").exists()]
    Yx = np.stack([np.load(CACHE_DIR / f"{d}_y.npy") for d in dd])
    folds = _folds_avail(MODEL); fp = None                 # 6-fold ensemble probs @ same THR
    for k in folds:
        pr, _ = _infer_ckpt(MODEL, _fold_ckpt(MODEL, k), dd)
        fp = pr if fp is None else fp + pr
    Mx = ((fp / max(1, len(folds))) >= THR).astype(np.float32)
    ts = pd.to_datetime(dd, format="%Y%m%d"); tset = set(ts)
    ww = u[u["source"].isin(["ff_warning", "fa_warning"])].copy()
    ww["issue_cdt"] = ww["issue_date"] - CDT; ww["expire_cdt"] = ww["expire_date"] - CDT
    ww = ww[(ww["expire_cdt"] >= ts.min()) &
            (ww["issue_cdt"] <= ts.max() + pd.Timedelta(days=1))]
    ww["day"] = [cdt_days(i, e) for i, e in zip(ww["issue_cdt"], ww["expire_cdt"])]
    evx = ww.explode("day", ignore_index=True); evx = evx[evx["day"].isin(tset)]
    jx = gpd.sjoin(cells_gdf, evx[["day", "geometry"]], predicate="intersects")
    wm = {}
    for day, g in jx.groupby("day"):
        a = np.zeros((GRID_R, GRID_C), np.float32); a[g["R"], g["C"]] = 1.0
        wm[day.strftime("%Y%m%d")] = a
    Wx = np.stack([wm.get(d, np.zeros((GRID_R, GRID_C), np.float32)) for d in dd])
    return dd, Yx, Mx, Wx


_B = cells_albers.total_bounds; BOXA = (_B[3] - _B[1]) / (_B[2] - _B[0])   # CONUS y/x aspect
# --- union of test + 2026 (2026 built once; rebuilt only if the threshold changes) ---
if globals().get("_u26_thr") != THR:
    _u26 = _build_2026(); _u26_thr = THR
days26, Y26, M26, W26 = _u26
n_te = len(te_days)
days_all = list(te_days) + list(days26)
Y_all = np.concatenate([Y, Y26])
M_all = np.concatenate([M, M26])
W_all = np.concatenate([W, W26])


def _src(i):
    return "test" if i < n_te else "2026"


def _fmt(s):
    return f"{s[4:6]}-{s[6:]}-{s[:4]}"          # US format MM-DD-YYYY


csi_m, csi_w = _day_csi(M_all, Y_all), _day_csi(W_all, Y_all)
# hand-picked days to display (in the union of test + 2026), left column then right
PICK = ["20230313", "20240109"]
idx = {d: i for i, d in enumerate(days_all)}
missing = [d for d in PICK if d not in idx]
if missing:
    raise ValueError(f"requested day(s) not in union: {missing}")
sel = [idx[d] for d in PICK]
print(f"union of {n_te} test + {len(days26)} 2026 days  |  showing {PICK}")
for di in sel:
    print(f"  {days_all[di]} ({_src(di)}): ConvGRU+Attn CSI {csi_m[di]:.2f}  vs  NWS CSI {csi_w[di]:.2f}")

# cols = the two chosen days (date header); rows = the two forecasts (own bg tint)
_B = cells_albers.total_bounds; BOXA = (_B[3] - _B[1]) / (_B[2] - _B[0])   # CONUS y/x aspect
cols = [(sel[0], "day0", days_all[sel[0]]), (sel[1], "day1", days_all[sel[1]])]
rows = [("ConvGRU+Attn", M_all, csi_m), ("NWS warnings", W_all, csi_w)]

# manual axes placement -> exactly equal horizontal & vertical gaps (in inches)
PW = 4.6; PH = PW * BOXA; G = 0.20                 # panel width/height (CONUS aspect) + equal gap
L, R, T, Bt = 0.45, 0.15, 0.5, 0.95                # margins (in): row-labels / right / titles / legend
FW = L + 2 * PW + G + R; FH = T + 2 * PH + G + Bt
fig = plt.figure(figsize=(FW, FH), dpi=300)                 # high-res (aspect unchanged)
fig.patch.set_facecolor("white")
axes = np.empty((2, 2), object)
for r in range(2):
    for c in range(2):
        x0 = (L + c * (PW + G)) / FW; y0 = (Bt + (1 - r) * (PH + G)) / FH
        axes[r, c] = fig.add_axes([x0, y0, PW / FW, PH / FH])
for r, (fname, Pp, csi_arr) in enumerate(rows):
    for c, (di, wname, dstr) in enumerate(cols):
        obs = (Y_all[di] > 0) & land
        pred = (Pp[di] > 0) & land
        cat = np.zeros((GRID_R, GRID_C))
        cat[obs & ~pred] = 1                               # miss
        cat[obs & pred] = 2                                # hit
        cat[pred & ~obs] = 3                               # false alarm
        gdf = cells_albers.copy()
        gdf["v"] = np.where(land, cat, np.nan)[RR, CC]
        ax = axes[r, c]
        ax.set_facecolor(ROW_BG[r])
        gdf.plot(column="v", cmap=CMAP, vmin=0, vmax=3, ax=ax, edgecolor="white", lw=0.15,
                 missing_kwds={"color": ROW_BG[r]})
        states.boundary.plot(ax=ax, color="0.5", lw=0.45, zorder=3)
        ax.set_xlim(_B[0], _B[2]); ax.set_ylim(_B[1], _B[3]); ax.set_aspect("auto")
        ax.set_xticks([]); ax.set_yticks([])
        for s_ in ax.spines.values():
            s_.set_visible(False)
        # per-panel scorecard (top-right, off the map)
        h = int((obs & pred).sum()); on = int(obs.sum())
        ax.text(0.90, 0.98, f"CSI {csi_arr[di]:.2f}\ncaught {h}/{on}",
                transform=ax.transAxes, va="top", ha="right", fontsize=10.5,
                fontweight="bold", color="#1a1a2e", linespacing=1.25,
                bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="0.8", alpha=0.92))
        if r == 0:
            ax.set_title(_fmt(dstr), fontsize=16, fontweight="bold", color="#1a1a2e", pad=6)
        if c == 0:
            ax.set_ylabel(fname, rotation=90, ha="center", va="center", fontsize=13,
                          fontweight="bold", color="#1a1a2e", labelpad=8)
fig.legend(handles=LEG, loc="lower center", ncol=2, fontsize=11.5, frameon=False,
           bbox_to_anchor=(0.5, (Bt * 0.22) / FH), handlelength=1.25, handleheight=1.25,
           columnspacing=1.6, handletextpad=0.6)
_poster = Path("poster_figs"); _poster.mkdir(exist_ok=True)
fig.savefig(_poster / "5_per_day_maps.png", dpi=300, facecolor="none",
            
            bbox_inches="tight", pad_inches=0.05)   # tight bbox -> legend not cropped
plt.show()

## 6. Aggregate spatial view

Summed over all test days: where floods were observed, where the model fired, and where NWS
warned - so persistent hits / gaps stand out geographically.

In [ ]:
# 6. Aggregate over all test days: observed floods vs model firings vs NWS warnings.
# Each cell's value = the number of the N test days it was hit; colour scale is shared.
agg = [("Observed floods", Y.sum(0), "#2a9d8f"),
       (f"{MODEL_NAME}", M.sum(0), "#e76f51"),
       ("NWS warnings", W.sum(0), "#6a4c93")]
ndays = len(te_days)
mx = max(float(a[land].max()) for _, a, _ in agg) or 1.0

# manual placement, cropped to the land extent -> maps big & tight (small equal gap)
_lm = land[RR, CC]
_LB = cells_albers[_lm].total_bounds                        # land-only bounds (drop ocean margin)
LA = (_LB[3] - _LB[1]) / (_LB[2] - _LB[0])                  # land y/x aspect
CMAP6 = "magma_r"                                           # yellow (low) -> purple (high)
PW = 6.0; PH = PW * LA; G = 0.12                            # panel width/height + gap (inches)
L, R, T = 0.1, 0.15, 0.55                                   # side/top margins (inches)
CBH, CB_GAP, CBL = 0.22, 0.22, 0.55                         # colorbar height / gap / label space
FW = L + 3 * PW + 2 * G + R; FH = CBL + CBH + CB_GAP + PH + T
fig = plt.figure(figsize=(FW, FH), dpi=200)                 # high-res
_ymaps = (CBL + CBH + CB_GAP) / FH
axmaps = [fig.add_axes([(L + i * (PW + G)) / FW, _ymaps, PW / FW, PH / FH]) for i in range(3)]
for ax, (title, a, col) in zip(axmaps, agg):
    gdf = cells_albers.copy()
    gdf["v"] = np.where(land, a, np.nan)[RR, CC]
    gdf.plot(column="v", cmap=CMAP6, ax=ax, vmin=0, vmax=mx, edgecolor="none",
             missing_kwds={"color": "none"})              # non-land transparent
    states.boundary.plot(ax=ax, color="0.4", lw=0.6, zorder=3)
    ax.set_xlim(_LB[0], _LB[2]); ax.set_ylim(_LB[1], _LB[3]); ax.set_aspect("auto")
    ax.set_axis_off()
    ax.set_title(f"{title}\nmax {int(a[land].max())} of {ndays} days",
                 fontsize=15, fontweight="bold", color=col, pad=10)
cax = fig.add_axes([L / FW, CBL / FH, (3 * PW + 2 * G) / FW, CBH / FH])   # horizontal, full width
cbar = fig.colorbar(plt.cm.ScalarMappable(cmap=CMAP6, norm=plt.Normalize(0, mx)),
                    cax=cax, orientation="horizontal")
cbar.set_label(f"days a cell was hit  (out of {ndays} test days)", fontsize=13, fontweight="bold")
cbar.ax.tick_params(labelsize=11)
_poster = Path("poster_figs"); _poster.mkdir(exist_ok=True)
fig.savefig(_poster / "6_aggregate_maps.png", dpi=300, facecolor="none")  # hi-res transparent
plt.show()

## 6b. Overlap with observed floods

One number: how much each predictor's per-cell flood-frequency map (days-flagged over the
test set) overlaps the **observed** one, on land cells. We use **Intersection-over-Union of
the day-counts** (weighted Jaccard):

`overlap = \u03a3 min(obs, pred) / \u03a3 max(obs, pred)`

`0` = no overlap, `1` = identical maps. It penalises both firing where floods didn't happen
and firing too much / too little. (For overlap on individual cell-**days** — space *and* time
— that is the CSI column in section 3.)